In [1]:
import os

repo_url = "https://github.com/nta2112/OW_OVD-An-custom.git"
working_dir = "/kaggle/working/OW_OVD"

if not os.path.exists(working_dir):
    print("Cloning repository...")
    !git clone {repo_url} {working_dir}
else:
    print("Repository đã tồn tại. Đang cập nhật (pull)...")
    %cd {working_dir}
    !git pull

%cd {working_dir}

Cloning repository...
Cloning into '/kaggle/working/OW_OVD'...
remote: Enumerating objects: 730, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 730 (delta 63), reused 71 (delta 34), pack-reused 627 (from 1)
Receiving objects: 100% (730/730), 1.36 MiB | 6.85 MiB/s, done.
Resolving deltas: 100% (458/458), done.
/kaggle/working/OW_OVD


In [2]:
import os
import sys

repo_root = os.path.abspath(os.getcwd())
repo_candidates = [
    repo_root,
    os.path.join(repo_root, "OW_OVD"),
    "/kaggle/working/OW_OVD",
    "/kaggle/working",
]

repo_root = None
for candidate in repo_candidates:
    if os.path.isdir(os.path.join(candidate, "yolo_world")):
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate the repo root containing the yolo_world package. "
        "Please set the working directory to the repo folder first."
    )

sys.path.insert(0, repo_root)
os.environ["PYTHONPATH"] = repo_root + (
    os.pathsep + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else ""
)
os.chdir(repo_root)

print("Changed working directory to:", os.getcwd())
print("Added repo root to PYTHONPATH:", repo_root)

Changed working directory to: /kaggle/working/OW_OVD
Added repo root to PYTHONPATH: /kaggle/working/OW_OVD


# Standalone visualization notebook for 10 different classes

This notebook is designed to be easy to rerun for paper evidence.

What it does:
- reads images from your dataset root
- picks up to 10 images from 10 different classes
- runs inference with your own config and checkpoint
- draws known boxes in yellow and unknown boxes in red

Dataset folder convention expected by this notebook:
- root/test/0/<class_name>/<image files>
- if your folder is flat, the notebook will automatically fall back to the first 10 files

In [3]:
import os
import sys
import glob
import importlib.util

# -------------------------------------------------------------------
# 0) Dùng đúng setup cũ từng chạy được trên Kaggle
# -------------------------------------------------------------------
repo_url = "https://github.com/nta2112/OW_OVD-An-custom.git"
working_dir = "/kaggle/working/OW_OVD"

if not os.path.exists(working_dir):
    print("Cloning repository...")
    !git clone {repo_url} {working_dir}
else:
    print("Repository already exists. Updating...")
    %cd {working_dir}
    !git pull

%cd {working_dir}

if not os.path.exists("third_party/mmyolo"):
    print("Cloning mmyolo...")
    !git clone https://github.com/open-mmlab/mmyolo.git third_party/mmyolo
else:
    print("mmyolo already exists.")

# -------------------------------------------------------------------
# 1) Cài đúng stack đã từng chạy thành công trên Kaggle
# -------------------------------------------------------------------
!pip install torch==2.4.0+cu121 torchvision==0.19.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121
!pip install mmcv -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html
!pip install matplotlib pycocotools terminaltables mmengine prettytable wcwidth open_clip_torch transformers
!pip install "mmdet>=3.1.0" --no-deps
!pip install --no-build-isolation --no-deps third_party/mmyolo

# -------------------------------------------------------------------
# 2) Vá lỗi version ceiling trong mmdet VÀ mmyolo nếu cần
# -------------------------------------------------------------------
def _patch_mmcv_ceiling(pkg_name, old_ceiling, new_ceiling="2.3.0"):
    spec = importlib.util.find_spec(pkg_name)
    if spec is None or not spec.origin:
        return
    init_file = spec.origin
    with open(init_file, "r", encoding="utf-8") as f:
        content = f.read()
    old_str = f"mmcv_maximum_version = '{old_ceiling}'"
    new_str = f"mmcv_maximum_version = '{new_ceiling}'"
    if old_str in content:
        content = content.replace(old_str, new_str)
        with open(init_file, "w", encoding="utf-8") as f:
            f.write(content)
        print(f"Patched {pkg_name}: ceiling {old_ceiling!r} -> {new_ceiling!r}")

# mmdet thường ceiling '2.2.0', mmyolo thường ceiling '2.1.0'
_patch_mmcv_ceiling("mmdet",  "2.2.0")
_patch_mmcv_ceiling("mmdet",  "2.1.0")
_patch_mmcv_ceiling("mmyolo", "2.1.0")
_patch_mmcv_ceiling("mmyolo", "2.2.0")

# Xóa cache module cũ để import lại sạch
_stale = [m for m in sys.modules if m.split(".")[0] in ("mmcv", "mmdet", "mmyolo", "mmengine")]
for _mod in _stale:
    del sys.modules[_mod]
if _stale:
    print("Evicted stale cached modules:", _stale)

# # -------------------------------------------------------------------
# # 3) Đảm bảo repo root nằm trên PYTHONPATH
# # -------------------------------------------------------------------
# repo_root = os.path.abspath(os.getcwd())
# if os.path.isdir(os.path.join(repo_root, "yolo_world")):
#     sys.path.insert(0, repo_root)
#     os.environ["PYTHONPATH"] = repo_root + (
#         os.pathsep + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else ""
#     )
#     print("Added repo root to PYTHONPATH:", repo_root)
# else:
#     print("WARNING: repo root does not contain yolo_world package:", repo_root)

# from PIL import Image, ImageDraw, ImageFont
# from mmdet.apis import init_detector, inference_detector

# # -------------------------------------------------------------------
# # 4) Chỉnh đường dẫn theo môi trường của bạn
# # -------------------------------------------------------------------
# TEST_ROOT = "/kaggle/input/datasets/rtlmhjbn/ip02-dataset/classification/test/0"
# CFG_PATH  = "configs/open_world/mowod/custom/ip102_t3.py"
# CKPT_PATH = "/kaggle/input/models/nta212/ow-ovd-checkpoint-task1-2-3-4/pytorch/default/1/t3_best.pth"
# OUT_DIR   = "/kaggle/working/OW_OVD/paper_visuals"

# os.makedirs(OUT_DIR, exist_ok=True)

# # -------------------------------------------------------------------
# # 5) Tự động tìm checkpoint nếu bạn đang dùng pattern *.pth
# # -------------------------------------------------------------------
# if "*" in CKPT_PATH:
#     candidates = sorted(glob.glob(CKPT_PATH, recursive=True))
#     if candidates:
#         CKPT_PATH = candidates[0]
#         print("Using checkpoint:", CKPT_PATH)
#     else:
#         raise FileNotFoundError(f"Không tìm thấy checkpoint theo pattern: {CKPT_PATH}")

# if not os.path.exists(TEST_ROOT):
#     raise FileNotFoundError(f"Không tìm thấy dataset root: {TEST_ROOT}")
# if not os.path.exists(CFG_PATH):
#     raise FileNotFoundError(f"Không tìm thấy config: {CFG_PATH}")
# if not os.path.exists(CKPT_PATH):
#     raise FileNotFoundError(f"Không tìm thấy checkpoint: {CKPT_PATH}")

# print("TEST_ROOT:", TEST_ROOT)
# print("CFG_PATH:",  CFG_PATH)
# print("CKPT_PATH:", CKPT_PATH)
# print("OUT_DIR:",   OUT_DIR)

Repository already exists. Updating...
/kaggle/working/OW_OVD
Already up to date.
/kaggle/working/OW_OVD
Cloning mmyolo...
Cloning into 'third_party/mmyolo'...
remote: Enumerating objects: 4968, done.
remote: Counting objects: 100% (1341/1341), done.
remote: Compressing objects: 100% (294/294), done.
remote: Total 4968 (delta 1133), reused 1047 (delta 1047), pack-reused 3627 (from 1)
Receiving objects: 100% (4968/4968), 3.62 MiB | 13.10 MiB/s, done.
Resolving deltas: 100% (3216/3216), done.
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 2.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 89.6 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 67.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 32.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 77.1 MB/

In [4]:
# -------------------------------------------------------------------
# 3) Đảm bảo repo root nằm trên PYTHONPATH
# -------------------------------------------------------------------
repo_root = os.path.abspath(os.getcwd())
if os.path.isdir(os.path.join(repo_root, "yolo_world")):
    sys.path.insert(0, repo_root)
    os.environ["PYTHONPATH"] = repo_root + (
        os.pathsep + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else ""
    )
    print("Added repo root to PYTHONPATH:", repo_root)
else:
    print("WARNING: repo root does not contain yolo_world package:", repo_root)

from PIL import Image, ImageDraw, ImageFont
from mmdet.apis import init_detector, inference_detector

# -------------------------------------------------------------------
# 4) Chỉnh đường dẫn theo môi trường của bạn
# -------------------------------------------------------------------
TEST_ROOT = "/kaggle/input/datasets/rtlmhjbn/ip02-dataset/classification/test/0"
CFG_PATH  = "configs/open_world/mowod/custom/ip102_t3.py"
CKPT_PATH = "/kaggle/input/models/nta212/ow-ovd-checkpoint-task1-2-3-4/pytorch/default/1/t3_best.pth"
OUT_DIR   = "/kaggle/working/OW_OVD/paper_visuals"

os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------------------------------------------------
# 5) Tự động tìm checkpoint nếu bạn đang dùng pattern *.pth
# -------------------------------------------------------------------
if "*" in CKPT_PATH:
    candidates = sorted(glob.glob(CKPT_PATH, recursive=True))
    if candidates:
        CKPT_PATH = candidates[0]
        print("Using checkpoint:", CKPT_PATH)
    else:
        raise FileNotFoundError(f"Không tìm thấy checkpoint theo pattern: {CKPT_PATH}")

if not os.path.exists(TEST_ROOT):
    raise FileNotFoundError(f"Không tìm thấy dataset root: {TEST_ROOT}")
if not os.path.exists(CFG_PATH):
    raise FileNotFoundError(f"Không tìm thấy config: {CFG_PATH}")
if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f"Không tìm thấy checkpoint: {CKPT_PATH}")

print("TEST_ROOT:", TEST_ROOT)
print("CFG_PATH:",  CFG_PATH)
print("CKPT_PATH:", CKPT_PATH)
print("OUT_DIR:",   OUT_DIR)

Added repo root to PYTHONPATH: /kaggle/working/OW_OVD


/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


TEST_ROOT: /kaggle/input/datasets/rtlmhjbn/ip02-dataset/classification/test/0
CFG_PATH: configs/open_world/mowod/custom/ip102_t3.py
CKPT_PATH: /kaggle/input/models/nta212/ow-ovd-checkpoint-task1-2-3-4/pytorch/default/1/t3_best.pth
OUT_DIR: /kaggle/working/OW_OVD/paper_visuals


In [ ]:
import os, json, glob, random

# -------------------------------------------------------------------
# Lay anh mau tu COCO detection test.json (co ground-truth bbox)
# -------------------------------------------------------------------
COCO_ANN_PATH = '/kaggle/input/datasets/eljazouly/ip102-coco-annotations/coco_annotations/test.json'
IMG_ROOT      = '/kaggle/input/datasets/rtlmhjbn/ip02-dataset/classification'
OUT_DIR       = '/kaggle/working/OW_OVD/paper_visuals'
os.makedirs(OUT_DIR, exist_ok=True)

with open(COCO_ANN_PATH, 'r') as f:
    coco = json.load(f)

id2fname = {img['id']: img['file_name'] for img in coco['images']}
annotated_ids = list({ann['image_id'] for ann in coco['annotations']})
random.seed(42)
random.shuffle(annotated_ids)

IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.bmp')

def find_image(file_name):
    direct = os.path.join(IMG_ROOT, file_name)
    if os.path.exists(direct):
        return direct
    basename = os.path.basename(file_name)
    for root, dirs, files in os.walk(IMG_ROOT):
        if basename in files:
            return os.path.join(root, basename)
    stem = os.path.splitext(basename)[0]
    for ext in IMAGE_EXTS:
        hits = glob.glob(os.path.join(IMG_ROOT, '**', stem + ext), recursive=True)
        if hits:
            return hits[0]
    return None

img2anns = {}
for ann in coco['annotations']:
    img2anns.setdefault(ann['image_id'], []).append(ann)

cat_id2name = {c['id']: c['name'] for c in coco['categories']}

sample_paths  = []
sample_gt_labels = []

for img_id in annotated_ids:
    if len(sample_paths) >= 10:
        break
    fname = id2fname[img_id]
    path  = find_image(fname)
    if path is None:
        continue
    anns = img2anns.get(img_id, [])
    if not anns:
        continue
    sample_paths.append(path)
    sample_gt_labels.append([cat_id2name.get(a['category_id'], '?') for a in anns])

if not sample_paths:
    print('WARNING: COCO images not found, falling back to classification images')
    fallback = '/kaggle/input/datasets/rtlmhjbn/ip02-dataset/classification/test'
    for root, dirs, files in os.walk(fallback):
        for fname in files:
            if any(fname.lower().endswith(e) for e in IMAGE_EXTS):
                sample_paths.append(os.path.join(root, fname))
            if len(sample_paths) >= 10:
                break
        if len(sample_paths) >= 10:
            break

print(f'Selected {len(sample_paths)} images from COCO detection test split:')
for i, p in enumerate(sample_paths, 1):
    lbls = sample_gt_labels[i-1] if i-1 < len(sample_gt_labels) else []
    print(f'  {i:02d} - {os.path.basename(p)} | GT: {set(lbls)}')


In [6]:
# Cell 5: Tạo thư mục, tải pretrain weights và sinh các file embeddings + XML annotations cần thiết
import json
import torch
import numpy as np
import os
from transformers import AutoTokenizer, CLIPTextModelWithProjection

# 1. Tạo các thư mục lưu dữ liệu
os.makedirs('pretrained_models', exist_ok=True)
os.makedirs('data/IP102', exist_ok=True)
os.makedirs('data/texts/IP102', exist_ok=True)

# 2. Tải pretrain weights của YOLO-World
print("-> Đang tải weights...")
weights_path = 'pretrained_models/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth'
if not os.path.exists(weights_path):
    !wget -O {weights_path} \
      https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth
else:
    print("Pretrained weights đã tồn tại.")

# 3. Đọc tên các class từ file annotations
ann_path = '/kaggle/input/datasets/eljazouly/ip102-coco-annotations/coco_annotations/train.json'
with open(ann_path, 'r') as f:
    coco_data = json.load(f)

categories = sorted(coco_data['categories'], key=lambda x: x['id'])
class_names = [cat['name'] for cat in categories]
num_classes = len(class_names)
print(f"Tổng số lớp học (classes): {num_classes}")

# 4. Lưu danh sách class vào file class_texts.json
class_texts = [[name] for name in class_names]
with open('data/texts/IP102/class_texts.json', 'w') as f:
    json.dump(class_texts, f)

# 5. Trích xuất text embeddings bằng CLIP
print("-> Đang sinh class embeddings từ CLIP model...")
model_name = 'openai/clip-vit-base-patch32'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = CLIPTextModelWithProjection.from_pretrained(model_name, use_safetensors=True)
model.eval()

embeddings = []
with torch.no_grad():
    for name in class_names:
        inputs = tokenizer(name, padding=True, return_tensors="pt")
        outputs = model(**inputs)
        embed = outputs.text_embeds[0].cpu().numpy()
        embed = embed / np.linalg.norm(embed)
        embeddings.append(embed)

np.save('data/IP102/ip102_gt_embeddings.npy', np.array(embeddings))

# 6. Sinh file task_att_1_embeddings.pth chứa cả att_embedding và att_text
print("-> Đang sinh file dummy attribute embeddings...")
num_att = num_classes * 25
torch.save({
    'att_embedding': torch.zeros(num_att, 512),
    'att_text': [f"att_{i}" for i in range(num_att)]
}, 'data/IP102/task_att_1_embeddings.pth')

# 7. Sinh file phân phối mẫu dummy
thrs = [0.55]
pos_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
neg_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
torch.save({
    'positive_distributions': pos_dist,
    'negative_distributions': neg_dist
}, 'data/IP102/mowod_distribution_sim1.pth')

# 8. Chuyển đổi COCO annotations sang PASCAL VOC (XML) siêu tốc phục vụ OWODEvaluator
print("-> Đang chuyển đổi COCO JSON sang cấu trúc PASCAL VOC...")
voc_root = 'data/IP102/voc/'
anno_dir = os.path.join(voc_root, "Annotations")
imageset_dir = os.path.join(voc_root, "ImageSets", "Main", "mowod")
os.makedirs(anno_dir, exist_ok=True)
os.makedirs(imageset_dir, exist_ok=True)

# Gom các annotation theo ID ảnh
image_annos = {}
for ann in coco_data['annotations']:
    image_annos.setdefault(ann['image_id'], []).append(ann)

# Tạo ánh xạ nhãn an toàn tự động phát hiện lệch chỉ số
cat_map = {cat['id']: cat['name'] for cat in categories}
anno_ids = {ann['category_id'] for ann in coco_data['annotations']}

if 0 in anno_ids and 0 not in cat_map:
    print("-> Phát hiện nhãn đánh số 0-indexed trong annotations. Đang tự động khớp chỉ số...")
    # Nếu nhãn đánh số từ 0 đến N-1
    if max(anno_ids) == len(class_names) - 1:
        cat_map = {i: class_names[i] for i in range(len(class_names))}
    else:
        # Dự phòng khẩn cấp
        cat_map[0] = class_names[0] if len(class_names) > 0 else 'unknown'
        for i in range(1, len(class_names) + 1):
            if i not in cat_map:
                cat_map[i] = class_names[i-1] if i-1 < len(class_names) else 'unknown'

image_names = []

for img in coco_data['images']:
    img_id = img['id']
    file_name = img['file_name']
    width = img.get('width', 640)
    height = img.get('height', 640)
    
    img_stem = os.path.splitext(os.path.basename(file_name))[0]
    image_names.append(img_stem)
    
    # Tạo chuỗi XML tối ưu
    xml_content = f"""<annotation>
  <filename>{os.path.basename(file_name)}</filename>
  <size>
    <width>{width}</width>
    <height>{height}</height>
    <depth>3</depth>
  </size>
"""
    annos = image_annos.get(img_id, [])
    for ann in annos:
        bbox = ann['bbox'] # [x, y, w, h]
        cat_name = cat_map.get(ann['category_id'], 'unknown')
        
        # Chuyển đổi [x, y, w, h] sang [xmin, ymin, xmax, ymax]
        xmin = max(1, int(round(bbox[0])))
        ymin = max(1, int(round(bbox[1])))
        xmax = max(xmin + 1, int(round(bbox[0] + bbox[2])))
        ymax = max(ymin + 1, int(round(bbox[1] + bbox[3])))
        
        xml_content += f"""  <object>
    <name>{cat_name}</name>
    <pose>Unspecified</pose>
    <truncated>0</truncated>
    <difficult>0</difficult>
    <bndbox>
      <xmin>{xmin}</xmin>
      <ymin>{ymin}</ymin>
      <xmax>{xmax}</xmax>
      <ymax>{ymax}</ymax>
    </bndbox>
  </object>
"""
    xml_content += "</annotation>"
    
    xml_path = os.path.join(anno_dir, f"{img_stem}.xml")
    with open(xml_path, 'w', encoding='utf-8') as f_xml:
        f_xml.write(xml_content)

# Ghi danh sách ảnh vào file
txt_path = os.path.join(imageset_dir, "all_task_test.txt")
with open(txt_path, 'w') as f_txt:
    f_txt.write("\n".join(image_names))

print(f"Thành công chuyển đổi {len(image_names)} nhãn sang VOC XML tại {voc_root}!")


-> Đang tải weights...
--2026-07-16 04:49:51--  https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth
Resolving huggingface.co (huggingface.co)... 18.65.14.100, 18.65.14.85, 18.65.14.87, ...
Connecting to huggingface.co (huggingface.co)|18.65.14.100|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cas-bridge.xethub.hf.co/xet-bridge-us/65bb7a71626a4c209906adf5/09dafb73b0d19d270cf20f7eeac6a7861303a753332d5df9917772ba23e4a47d?Expires=1784180992&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzY1YmI3YTcxNjI2YTRjMjA5OTA2YWRmNS8wOWRhZmI3M2IwZDE5ZDI3MGNmMjBmN2VlYWM2YTc4NjEzMDNhNzUzMzMyZDVkZjk5MTc3NzJiYTIzZTRhNDdkKiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4NDE4MDk5Mn19fV19&Signature=MEYCIQD5MnuB90qUjpq6FhMqbYHxSxE2%7EH-1ehskbdkNPXv5ZAIhAJIyrJgCk4GZ1nmPIlVlRX1unLngiZnAPWQ7mdkdp3RB&Key-Pair-Id=K3EPXBYC3CKDRZ&X-Xet-Cas-Uid

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.la

-> Đang sinh file dummy attribute embeddings...
-> Đang chuyển đổi COCO JSON sang cấu trúc PASCAL VOC...
-> Phát hiện nhãn đánh số 0-indexed trong annotations. Đang tự động khớp chỉ số...
Thành công chuyển đổi 45095 nhãn sang VOC XML tại data/IP102/voc/!


In [ ]:
# -------------------------------------------------------------------
# Load model  –  Two-stage checkpoint loading
#
# WHY TWO STAGES?
# t3_best.pth is an INCREMENTAL checkpoint – it only contains weights
# that changed during Task-3 training (embeddings + att_embeddings).
# The bbox regression head (reg_preds / cls_preds) was FROZEN and is
# NOT present in t3_best.pth.  Those weights live in t1_best.pth.
#
# Loading order:
#   1. t1_best.pth  – full backbone + neck + head (reg_preds, cls_preds …)
#   2. t3_best.pth  – overwrites embeddings + att_embeddings with Task-3 values
# -------------------------------------------------------------------
import re, os, sys, glob, importlib, importlib.util, importlib.metadata, torch

# ── Patch mmcv version ceiling ────────────────────────────────────
def _patch_ceiling(pkg_name):
    importlib.invalidate_caches()
    spec = importlib.util.find_spec(pkg_name)
    if not (spec and spec.origin): return
    path = spec.origin
    src  = open(path, encoding="utf-8").read()
    m = re.search(r"mmcv_maximum_version\s*=\s*[\"']([^\"']+)[\"']", src)
    if not m: return
    print(f"[patch] {pkg_name}: ceiling = {m.group(1)!r}  ({path})")
    new_src = re.sub(r"(mmcv_maximum_version\s*=\s*)[\"'][^\"']+[\"']", r"\g<1>'2.3.0'", src)
    if new_src != src:
        open(path, "w", encoding="utf-8").write(new_src)
    else:
        print(f"[patch] {pkg_name}: already '2.3.0'")
    cache_dir = os.path.join(os.path.dirname(path), "__pycache__")
    for pyc in glob.glob(os.path.join(cache_dir, "__init__*.pyc")):
        try: os.remove(pyc); print(f"[patch] removed .pyc: {pyc}")
        except OSError: pass

_patch_ceiling("mmdet")
_patch_ceiling("mmyolo")

# ── Evict stale modules ───────────────────────────────────────────
_stale = [m for m in sys.modules
          if m.split(".")[0] in ("mmcv","mmdet","mmyolo","mmengine","yolo_world")]
for _m in _stale: del sys.modules[_m]
importlib.invalidate_caches()
print(f"[patch] Evicted {len(_stale)} stale modules")

# ── Ensure repo root on sys.path ─────────────────────────────────
_repo_root = os.path.abspath(os.getcwd())
for _c in [_repo_root, "/kaggle/working/OW_OVD", "/kaggle/working"]:
    if os.path.isdir(os.path.join(_c, "yolo_world")):
        _repo_root = _c; break
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)
print(f"[patch] repo_root = {_repo_root}")

import mmcv, mmyolo  # noqa
print("mmcv:", mmcv.__version__)
print("mmyolo:", importlib.metadata.version("mmyolo"))

import yolo_world, yolo_world.models  # noqa – registers OurDetector, OurHead, etc.
print("[ok] yolo_world registered into mmyolo registry")

from mmdet.apis import init_detector
from mmengine.runner import load_checkpoint
from mmengine.config import Config
from PIL import Image, ImageDraw, ImageFont

# ── Paths ─────────────────────────────────────────────────────────
CKPT_DIR = "/kaggle/input/models/nta212/ow-ovd-checkpoint-task1-2-3-4/pytorch/default/1"
T1_CKPT  = os.path.join(CKPT_DIR, "t1_best.pth")   # full weights
T3_CKPT  = os.path.join(CKPT_DIR, "t3_best.pth")   # task-3 delta

# ── Patch torch.load (weights_only default) ───────────────────────
if not getattr(torch.load, "_owd_safe_patch", False):
    _orig_load = torch.load
    def _safe_load(*a, **kw): kw.setdefault("weights_only", False); return _orig_load(*a, **kw)
    _safe_load._owd_safe_patch = True
    _safe_load._owd_original   = _orig_load
    torch.load = _safe_load

# ── Stage 0: init architecture ────────────────────────────────────
print("\nInitializing model architecture...")
model = init_detector(CFG_PATH, checkpoint=None, device="cpu")

# ── Stage 1: load t1_best.pth – full head weights ─────────────────
print(f"\n[Stage 1] Loading FULL weights from: {T1_CKPT}")
load_checkpoint(model, T1_CKPT, map_location="cpu")

# Sanity check: reg_preds should now have non-random values
for name, param in model.named_parameters():
    if "reg_pred" in name and "weight" in name:
        print(f"  reg_pred sanity – mean={param.data.mean():.4f}, std={param.data.std():.4f}")
        break
else:
    print("  WARNING: no reg_pred parameter found – check model architecture")

# ── Resize att_embeddings → 1925 before loading t3 ───────────────
if hasattr(model, "bbox_head") and hasattr(model.bbox_head, "select_att"):
    print("\nCalling select_att() to adjust att_embeddings size...")
    model.bbox_head.select_att(per_class=25)
    print("att_embeddings shape:", model.bbox_head.att_embeddings.shape)

# ── Stage 2: overlay t3_best.pth – embeddings + att_embeddings ───
print(f"\n[Stage 2] Loading Task-3 delta weights from: {T3_CKPT}")
t3_sd = torch.load(T3_CKPT, map_location="cpu")
t3_sd = t3_sd.get("state_dict", t3_sd)

result = model.load_state_dict(t3_sd, strict=False)
applied = [k for k in t3_sd.keys() if k not in result.missing_keys]
print(f"[Stage 2] Applied {len(applied)} keys: {applied[:10]}")
if result.unexpected_keys:
    print(f"[Stage 2] Unexpected keys (ignored): {result.unexpected_keys[:5]}")

# ── Force correct class names from config ─────────────────────────
cfg = Config.fromfile(CFG_PATH)
if hasattr(cfg, "class_names"):
    active_classes = list(cfg.class_names)[:77]
else:
    active_classes = [f"pest_{i}" for i in range(77)]
model.dataset_meta = {"classes": active_classes}

class_names = active_classes
UNKNOWN_ID  = 80
model.eval()

print(f"\nModel ready. {len(class_names)} classes loaded.")
print(f"embeddings     : {model.embeddings.shape}")
print(f"att_embeddings : {model.bbox_head.att_embeddings.shape}")


In [ ]:
from mmdet.apis import inference_detector
import json, os
from PIL import Image, ImageDraw, ImageFont
import numpy as np

# NOTE: IP102 COCO annotations use full-image boxes (classification-as-detection).
# The model correctly predicts full-image boxes + class label.
# Visualization: show top prediction as a text banner on the image.

SCORE_THR = 0.10

# Load GT labels from test.json
COCO_ANN_PATH = '/kaggle/input/datasets/eljazouly/ip102-coco-annotations/coco_annotations/test.json'
with open(COCO_ANN_PATH, 'r') as f:
    coco_data = json.load(f)
cat2name = {c['id']: c['name'] for c in coco_data['categories']}
img2ann = {}
for ann in coco_data['annotations']:
    img2ann.setdefault(ann['image_id'], []).append(ann)
fname2gt = {}
for img in coco_data['images']:
    anns = img2ann.get(img['id'], [])
    bn = os.path.basename(img['file_name'])
    gt_labels = list({cat2name.get(a['category_id'], '?') for a in anns})
    fname2gt[bn] = gt_labels

try:
    font_lg = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 24)
    font_sm = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 18)
except Exception:
    font_lg = font_sm = ImageFont.load_default()

model.eval()
for idx, img_path in enumerate(sample_paths, start=1):
    result = inference_detector(model, img_path)
    preds  = result.pred_instances
    scores = preds.scores.cpu().numpy()
    labels = preds.labels.cpu().numpy().astype(int)

    img  = Image.open(img_path).convert('RGB')
    W, H = img.width, img.height
    basename = os.path.basename(img_path)
    gt_labels = fname2gt.get(basename, [])

    # Pick top prediction
    if len(scores) > 0:
        top_idx   = int(np.argmax(scores))
        top_score = float(scores[top_idx])
        top_lid   = int(labels[top_idx])
        is_unknown = top_lid >= len(class_names)
        pred_name  = 'unknown' if is_unknown else class_names[top_lid]
    else:
        top_score = 0.0
        pred_name = 'no detection'
        is_unknown = False

    # Draw a thick colored border around the image
    border = 6
    border_color = (255, 50, 50) if is_unknown else (255, 215, 0)
    draw = ImageDraw.Draw(img)
    for b in range(border):
        draw.rectangle([b, b, W-1-b, H-1-b], outline=border_color)

    # Draw prediction banner at the top
    banner_h = 36
    banner = Image.new('RGB', (W, banner_h), color=(30, 30, 30))
    bdraw  = ImageDraw.Draw(banner)
    pred_txt = f'PRED: {pred_name}  ({top_score:.2f})'
    bdraw.text((8, 6), pred_txt, fill=border_color, font=font_sm)

    # Draw GT banner at the bottom
    gt_banner = Image.new('RGB', (W, 30), color=(20, 60, 20))
    gdraw     = ImageDraw.Draw(gt_banner)
    gt_txt    = 'GT: ' + ', '.join(gt_labels) if gt_labels else 'GT: (none)'
    gdraw.text((8, 5), gt_txt, fill=(100, 255, 100), font=font_sm)

    # Stack: top banner + image + bottom banner
    total_h = banner_h + H + 30
    canvas  = Image.new('RGB', (W, total_h), color=(20, 20, 20))
    canvas.paste(banner, (0, 0))
    canvas.paste(img,    (0, banner_h))
    canvas.paste(gt_banner, (0, banner_h + H))

    correct = any(pred_name.lower() in g.lower() or g.lower() in pred_name.lower()
                  for g in gt_labels)
    out_path = os.path.join(OUT_DIR, f'pred_{idx:02d}.png')
    canvas.save(out_path)
    status = 'CORRECT' if correct else 'WRONG'
    print(f'Image {idx}: {basename}  PRED={pred_name}({top_score:.2f})  GT={gt_labels}  [{status}]')

print('\nDone. Yellow border=Known, Red border=Unknown')
print('Top banner=prediction, Bottom banner=ground truth')


In [ ]:
# -------------------------------------------------------------------
# Trực quan hóa bounding box gốc (Ground Truth) từ COCO Annotations
# -------------------------------------------------------------------
import os
import json
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# 1. Định nghĩa đường dẫn ảnh và file annotations COCO tương ứng
image_path = "/kaggle/input/datasets/rtlmhjbn/ip02-dataset/classification/train/0/00003.jpg"
coco_json_path = "/kaggle/input/datasets/eljazouly/ip102-coco-annotations/coco_annotations/train.json"

# 2. Đọc file COCO JSON
print("Đang đọc file COCO JSON...")
with open(coco_json_path, 'r', encoding='utf-8') as f:
    coco_data = json.load(f)

# Tạo dictionary mapping để tra cứu nhanh
categories = {cat['id']: cat['name'] for cat in coco_data['categories']}
images = coco_data['images']
annotations = coco_data['annotations']

# 3. Tìm image_id dựa trên tên file (so sánh basename đề phòng lệch đường dẫn)
target_filename = os.path.basename(image_path)
image_id = None
image_info = None

for img in images:
    if os.path.basename(img['file_name']) == target_filename:
        image_id = img['id']
        image_info = img
        break

if image_id is None:
    raise ValueError(f"Không tìm thấy ảnh {target_filename} trong file COCO JSON.")

print(f"Tìm thấy ảnh: {image_info['file_name']} (ID: {image_id})")

# 4. Lấy tất cả bounding box (annotations) của ảnh này
img_annos = [ann for ann in annotations if ann['image_id'] == image_id]
print(f"Số lượng bounding box: {len(img_annos)}")

# 5. Vẽ bounding box lên ảnh
if not os.path.exists(image_path):
    print(f"Cảnh báo: Không tìm thấy file ảnh tại: {image_path} trên máy cục bộ này.")
    print("Nhưng code cell này đã sẵn sàng để bạn chạy trực tiếp trên Kaggle!")
else:
    img = Image.open(image_path)
    fig, ax = plt.subplots(1, figsize=(10, 10))
    ax.imshow(img)

    for ann in img_annos:
        bbox = ann['bbox'] # COCO format: [x_min, y_min, width, height]
        x, y, w, h = bbox
        cat_name = categories.get(ann['category_id'], f"Class {ann['category_id']}")
        
        # Tạo hình chữ nhật vẽ bbox
        rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        
        # Thêm text hiển thị nhãn của class
        plt.text(x, y - 5, cat_name, color='white', fontsize=12, bbox=dict(facecolor='red', alpha=0.5))

    plt.axis('off')
    plt.show()
